In [1]:
# %matplotlib inline
%config InlineBackend.figure_format = "retina"
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
})

import seaborn as sns
sns.set_theme(context="talk", style="whitegrid", 
              palette="colorblind", color_codes=True, 
              rc={"figure.figsize": [12, 8]})

import yfinance as yf
import numpy as np
import pandas as pd
import quantstats as qs

import requests
import csv

from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

import statsmodels.api as sm

In [2]:
# Carteira teorica do Idiv

url = "https://raw.githubusercontent.com/BDonadelli/Finance-playground/main/data/Cart_Idiv.csv"

response = requests.get(url)
response.encoding = 'utf-8'  # ou 'latin-1' se necessário

# Divide o conteúdo em linhas
linhas = response.text.splitlines()

# Ignora as duas primeiras e duas últimas linhas
linhas_filtradas = linhas[2:-2]

# Extrai a primeira coluna de cada linha restante
ASSETS = []
for linha in linhas_filtradas:
    # O separador é ";", pega o primeiro campo
    campos = linha.split(';')
    if campos:  # garante que a linha não está vazia
        ASSETS.append((campos[0],campos[4]))

ASSETS = [(ticker, float(str(value).replace(',', '.'))) for ticker, value in ASSETS]

try:
    pesos_indice = [value for ticker, value in ASSETS]
    tickers = [ticker+'.SA' for ticker, value in ASSETS]
    tickers.sort()
except :    
    tickers = [ticker+'.SA' for ticker in ASSETS]

tickers.sort()

print(tickers)

['ABCB4.SA', 'AGRO3.SA', 'ALOS3.SA', 'BBAS3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BRAP4.SA', 'BRBI11.SA', 'BRSR6.SA', 'CMIG4.SA', 'CMIN3.SA', 'CPFE3.SA', 'CSMG3.SA', 'CURY3.SA', 'CXSE3.SA', 'DIRR3.SA', 'EGIE3.SA', 'EVEN3.SA', 'EZTC3.SA', 'FESA4.SA', 'FLRY3.SA', 'GRND3.SA', 'ISAE4.SA', 'ITSA4.SA', 'ITUB3.SA', 'ITUB4.SA', 'JHSF3.SA', 'KEPL3.SA', 'KLBN11.SA', 'LAVV3.SA', 'LEVE3.SA', 'LOGG3.SA', 'MBRF3.SA', 'ODPV3.SA', 'PETR3.SA', 'PETR4.SA', 'PGMN3.SA', 'POMO4.SA', 'RANI3.SA', 'RECV3.SA', 'SAPR11.SA', 'SLCE3.SA', 'SYNE3.SA', 'TAEE11.SA', 'TGMA3.SA', 'TIMS3.SA', 'UNIP6.SA', 'VALE3.SA', 'VBBR3.SA', 'VLID3.SA', 'VULC3.SA']


#### Parâmetros

In [3]:
rf = 0.14               # taxa livre de risco
n_days=252              # dias no ano do calendario financeiro, assumindo dados diários pegos no Yahoo Finance
#n_monte_carlo = 10**6 # quantidade de carteiras na simulação

# Definição do período e download dos dados
data_inicio = '2018-01-01'
data_fim = '2026-04-30'

#### preços de fechamento

Baixa dados e limpa a base

In [4]:
prices = yf.download(tickers, start=data_inicio, end=data_fim , auto_adjust=True)['Close']
prices.columns = [col.replace('.SA', '') for col in prices.columns]

benchm = yf.download('^BVSP', start=data_inicio, end=data_fim , auto_adjust=True)['Close']

[*********************100%***********************]  52 of 52 completed
[*********************100%***********************]  1 of 1 completed


In [5]:
# Empresas com mais de 'limiar' dados faltantes
limiar = 10
missing = prices.isna().sum()
empresas_missing = missing[missing > limiar].index.tolist()

print(missing[missing > limiar].sort_values(ascending=False))

tickers = [item for item in ASSETS if item not in empresas_missing]



BRBI11    872
RECV3     824
CXSE3     821
CMIN3     774
CURY3     674
LAVV3     663
PGMN3     663
ALOS3     394
LOGG3     242
VBBR3      49
dtype: int64


In [6]:
import plotly.express as px

if empresas_missing:
    fig = px.line(
        prices[empresas_missing].reset_index(),
        x='Date',
        y=empresas_missing,
        title='Empresas com mais de 10 dados faltantes',
        labels={'value': 'Preço', 'Date': 'Data', 'variable': 'Empresa'}
    )

    fig.show()

In [7]:
# mantem colunas (axis=1) ue possuem no mínimo len(prices) - 20 valores não-nulos.
prices = prices.dropna(axis=1, thresh=len(prices) - 10)
# preenche dados faltantes repetindo ultimo valor
prices = prices.ffill()

prices

,ABCB4,AGRO3,BBAS3,BBDC3,BBDC4,BBSE3,BRAP4,BRSR6,CMIG4,CPFE3,...,SAPR11,SLCE3,SYNE3,TAEE11,TGMA3,TIMS3,UNIP6,VALE3,VLID3,VULC3
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,9.208960,6.842369,9.178535,11.138404,12.179823,13.367785,4.202160,7.811342,1.498885,10.347330,...,12.398583,3.451245,0.866539,9.415704,11.868074,7.978253,5.058343,21.669315,12.036351,5.375476
2018-01-03,9.247840,6.836968,9.295599,11.185791,12.235808,13.377100,4.241884,7.848087,1.485963,10.219520,...,12.291393,3.481202,0.913177,9.428867,11.868074,7.984299,5.428318,21.539461,12.218150,5.641645
2018-01-04,9.220070,7.058386,9.384792,11.389476,12.436577,13.405047,4.362472,7.895333,1.468735,10.144962,...,12.192606,3.467473,0.951421,9.279690,12.324995,7.948032,5.370700,21.627764,12.218150,5.786304
2018-01-05,9.353371,7.204199,9.384792,11.392922,12.507012,13.493544,4.467455,7.998998,1.470888,9.979877,...,12.285088,3.532378,0.951421,9.323567,12.461479,8.014519,5.455612,21.965372,12.193076,5.786304
2018-01-08,9.442240,7.236601,9.407089,11.392922,12.503490,13.572725,4.545484,8.132941,1.475195,10.139639,...,12.211524,3.513654,0.960748,9.279690,12.461479,7.905726,5.701251,22.453604,12.161732,5.815236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,25.160000,19.680000,23.000000,17.191864,19.949961,34.270000,24.059999,16.155130,13.088566,50.148193,...,42.849998,17.530001,4.060000,43.706520,32.939999,26.049999,60.869999,85.970001,19.760000,16.120001
2026-04-24,25.280001,19.850000,22.700001,17.091970,19.900011,34.290001,23.950001,15.876423,12.804031,49.388653,...,43.610001,17.370001,4.000000,43.168262,32.230000,26.100000,60.540001,85.870003,19.850000,15.940000
2026-04-27,24.930000,19.139999,22.510000,16.952118,19.710201,33.950001,23.850000,15.687300,12.558743,48.906994,...,41.549999,17.240000,3.940000,42.678936,32.250000,25.770000,59.880001,85.500000,19.440001,15.780000


In [8]:
returns = prices.pct_change().dropna()

print(len(prices),len(returns),len(tickers),len(ASSETS),len(prices.columns))

2068 2067 52 52 42


Matriz covariância por Ledoit Wolf

In [9]:
from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

"""
cov_matrix é uma NxN matriz de covariância
tickers é um dicionário com os ativos
"""

pesos = [item[1] / 100 for item in ASSETS]
#dicionario_pesos = dict(zip(tickers, pesos))

#cov = np.cov(market_prices)

model = LedoitWolf()
cov_matrix = model.fit(returns).covariance_
df_cov_matrix = pd.DataFrame(cov_matrix, index=returns.columns, columns=returns.columns)

print(df_cov_matrix.shape)
#print(dicionario_pesos)
print(tickers)

tickers_validos = list(df_cov_matrix.columns)
dicionario_pesos = {}
lista_tickers = []

for ticker, peso in tickers:
    if ticker in tickers_validos:
        dicionario_pesos[ticker] = peso
        lista_tickers.append(ticker)

print(lista_tickers)

(42, 42)
[('ABCB4', 0.449), ('ALOS3', 3.005), ('BRSR6', 0.769), ('BBSE3', 4.582), ('BRBI11', 0.195), ('BBDC3', 4.169), ('BBDC4', 3.993), ('BRAP4', 1.248), ('BBAS3', 4.074), ('AGRO3', 0.279), ('CXSE3', 2.357), ('CMIG4', 5.169), ('CSMG3', 2.324), ('CPFE3', 1.967), ('CMIN3', 1.724), ('CURY3', 1.237), ('DIRR3', 0.926), ('EGIE3', 2.53), ('EVEN3', 0.28), ('EZTC3', 0.366), ('FESA4', 0.261), ('FLRY3', 1.511), ('GRND3', 0.243), ('RANI3', 0.2), ('ISAE4', 2.416), ('ITSA4', 3.222), ('ITUB3', 4.026), ('ITUB4', 3.28), ('JHSF3', 0.582), ('KEPL3', 0.303), ('KLBN11', 3.097), ('LAVV3', 0.226), ('LOGG3', 0.302), ('POMO4', 0.915), ('MBRF3', 3.394), ('LEVE3', 0.395), ('ODPV3', 0.736), ('PGMN3', 0.332), ('PETR3', 6.816), ('PETR4', 6.905), ('RECV3', 0.801), ('SAPR11', 1.952), ('SLCE3', 0.874), ('SYNE3', 0.052), ('TAEE11', 1.991), ('TGMA3', 0.211), ('TIMS3', 3.753), ('UNIP6', 0.637), ('VALE3', 4.296), ('VLID3', 0.322), ('VBBR3', 3.898), ('VULC3', 0.408)]
['ABCB4', 'BRSR6', 'BBSE3', 'BBDC3', 'BBDC4', 'BRAP4', 

In [10]:
delta = black_litterman.market_implied_risk_aversion(prices)
prior = black_litterman.market_implied_prior_returns(dicionario_pesos, delta, cov_matrix)

print(delta)

ABCB4     1.777735
AGRO3     2.011665
BBAS3     1.329574
BBDC3     0.932728
BBDC4     0.966698
BBSE3     2.321506
BRAP4     2.204130
BRSR6     1.226213
CMIG4     2.666632
CPFE3     3.149688
CSMG3     2.605376
DIRR3     2.237331
EGIE3     2.913168
EVEN3     0.752133
EZTC3     0.630332
FESA4     1.167987
FLRY3     0.321384
GRND3     0.549470
ISAE4     3.358213
ITSA4     2.427834
ITUB3     2.534812
ITUB4     1.915717
JHSF3     1.851543
KEPL3     1.938175
KLBN11    1.258539
LEVE3     1.699174
MBRF3     1.234643
ODPV3     0.983800
PETR3     2.150487
PETR4     2.226396
POMO4     1.389336
RANI3     1.247864
SAPR11    2.085161
SLCE3     2.054292
SYNE3     1.155593
TAEE11    5.249974
TGMA3     1.234114
TIMS3     2.061351
UNIP6     2.270383
VALE3     1.759532
VLID3     0.749243
VULC3     1.335835
dtype: float64


/home/caio/Documentos/ic_26/.venv/lib/python3.12/site-packages/pypfopt/black_litterman.py:45: RuntimeWarning: If cov_matrix is not a dataframe, market cap index must be aligned to cov_matrix
  warnings.warn(


In [11]:
print(prior)

ABCB4     0.000302
AGRO3     0.000187
BBAS3     0.000311
BBDC3     0.000209
BBDC4     0.000219
BBSE3     0.000275
BRAP4     0.000341
BRSR6     0.000234
CMIG4     0.000558
CPFE3     0.000424
CSMG3     0.000446
DIRR3     0.000490
EGIE3     0.000346
EVEN3     0.000199
EZTC3     0.000166
FESA4     0.000171
FLRY3     0.000052
GRND3     0.000083
ISAE4     0.000335
ITSA4     0.000455
ITUB3     0.000445
ITUB4     0.000372
JHSF3     0.000469
KEPL3     0.000224
KLBN11    0.000102
LEVE3     0.000266
MBRF3     0.000200
ODPV3     0.000111
PETR3     0.000456
PETR4     0.000467
POMO4     0.000309
RANI3     0.000261
SAPR11    0.000311
SLCE3     0.000196
SYNE3     0.000188
TAEE11    0.000486
TGMA3     0.000268
TIMS3     0.000296
UNIP6     0.000391
VALE3     0.000272
VLID3     0.000148
VULC3     0.000261
dtype: float64


In [14]:
fatores_nefin = pd.read_csv('nefin_factors.csv')

# A coluna Date já existe, só converter e setar como índice
fatores_nefin['Date'] = pd.to_datetime(fatores_nefin['Date'])
fatores_nefin.set_index('Date', inplace=True)

# Reamostrar os fatores diários para mensais
fatores_mensais = fatores_nefin.resample('ME').sum()

# Alinhar com os retornos
datas_comuns = returns.index.intersection(fatores_mensais.index)
retornos_alinhados = returns.loc[datas_comuns]
fatores_alinhados = fatores_mensais.loc[datas_comuns]

In [19]:

col_rf = 'Risk_Free'

retornos_esperados_ff = pd.Series(index=retornos_alinhados.columns, dtype=float)

X = sm.add_constant(fatores_alinhados)  # Fatores + intercepto
medias_fatores = fatores_alinhados.drop(columns=[col_rf]).mean()
rf_media = fatores_alinhados[col_rf].mean()

for ticker in retornos_alinhados.columns:
    # Y = excesso de retorno do ativo (retorno - taxa livre de risco)
    Y = retornos_alinhados[ticker] - fatores_alinhados[col_rf]

    # Regressão OLS
    modelo = sm.OLS(Y, X).fit()

    # r_esperado = rf + alpha + sum(beta_fator * media_fator)
    fatores_cols = [c for c in fatores_alinhados.columns if c != col_rf]
    retorno_estimado = (
        rf_media
        + modelo.params['const']
        + sum(modelo.params[f] * medias_fatores[f] for f in fatores_cols)
    )

    retornos_esperados_ff[ticker] = retorno_estimado

print(retornos_esperados_ff.sort_values(ascending=False))

AGRO3     0.016473
RANI3     0.013130
KEPL3     0.012258
SLCE3     0.012063
FESA4     0.011025
PETR3     0.010988
MBRF3     0.010395
PETR4     0.008447
KLBN11    0.007014
ISAE4     0.006985
VALE3     0.006221
TIMS3     0.003871
BRAP4     0.002978
CPFE3     0.001518
TAEE11    0.001151
EGIE3     0.000839
FLRY3    -0.001803
VLID3    -0.002235
GRND3    -0.002281
CMIG4    -0.002953
UNIP6    -0.003644
SAPR11   -0.005427
TGMA3    -0.005749
ODPV3    -0.006074
CSMG3    -0.007950
LEVE3    -0.008245
ABCB4    -0.008697
VULC3    -0.008867
SYNE3    -0.008976
EVEN3    -0.009321
POMO4    -0.009555
EZTC3    -0.009687
ITSA4    -0.012565
BBAS3    -0.013501
ITUB3    -0.013887
BBSE3    -0.016614
ITUB4    -0.017397
BRSR6    -0.018906
BBDC3    -0.019094
DIRR3    -0.020302
BBDC4    -0.022389
JHSF3    -0.023139
dtype: float64
